# Cobertura de médico de família em Portugal Continental

**Fase 1 — Entendimento do Negócio | CRISP-DM**

O projeto analisa as diferenças de cobertura de médico de família entre unidades de saúde e a sua evolução entre janeiro de 2024 e junho de 2026. O objetivo é transformar os dados públicos do SNS numa leitura comparável da situação e avaliar se o histórico ajuda a antecipar o indicador do mês seguinte.

| Campo | Definição |
|---|---|
| Área | Acesso aos cuidados de saúde primários |
| Natureza | Análise independente baseada em dados públicos |
| Destinatários potenciais | Equipas de acompanhamento e planeamento em saúde e pessoas interessadas em consultar os indicadores |
| Fonte principal | Portal da Transparência do SNS |
| Âmbito territorial da análise principal | Portugal Continental, por ULS e região de saúde |
| Histórico a preservar na recolha | Janeiro de 2016 a junho de 2026 |
| Período da análise principal | Janeiro de 2024 a junho de 2026, sujeito à comparabilidade das unidades |
| Data do enquadramento | 13 de outubro de 2026 |

O CRISP-DM organiza o trabalho em seis fases: entendimento do negócio, entendimento dos dados, preparação, modelação, avaliação e disponibilização. Este notebook regista o problema, o âmbito, as hipóteses e os critérios de sucesso. A evidência sobre os dados será reproduzida na fase 2 e as decisões de preparação serão implementadas na fase 3.

## 1. Contexto

O Portal da Transparência do SNS publica o número de inscritos nos cuidados de saúde primários, a atribuição de médico de família e indicadores de utilização de consultas, por período e unidade de saúde.

Estes dados permitem acompanhar a cobertura, mas uma comparação baseada apenas no número de inscritos sem médico favorece as unidades maiores. É necessário observar em conjunto o número de pessoas afetadas, a percentagem de inscritos e a evolução da situação.

O universo analisado é o dos **utentes inscritos** nas unidades abrangidas pela fonte, não a população residente de cada território. A fonte refere Portugal Continental e exclui inscritos em postos empresa.

A análise usa o conjunto [Utentes Inscritos em Cuidados de Saúde Primários](https://transparencia.sns.gov.pt/explore/assets/utentes-inscritos-em-cuidados-de-saude-primarios/). Os valores e definições serão conferidos na fase de entendimento dos dados.

### 1.1. Periodicidade e significado dos valores

Os metadados do indicador antigo identificam periodicidade **mensal** e cobertura de janeiro de 2016 a junho de 2026. Uma linha corresponde à situação de uma unidade num mês de referência.

As contagens e percentagens de atribuição de médico referem-se aos inscritos nesse mês. As taxas de utilização de consultas são identificadas como indicadores de **um ano**, não como consultas realizadas apenas nesse mês. Os limites exatos dessa janela serão confirmados antes de interpretar ou usar essas taxas.

### 1.2. Justificação do período de análise

O projeto preserva o histórico completo na recolha, mas concentra a análise por unidade em **janeiro de 2024 a junho de 2026**. O recorte decorre de duas mudanças diferentes:

| Marco | Evidência | Consequência para a análise |
|---|---|---|
| Janeiro de 2024 | Reorganização dos ACES e outras entidades em ULS, com integração e transferência de unidades. | Não ligar diretamente os antigos ACES às atuais ULS sem verificar os territórios abrangidos. |
| Julho de 2026 | Alteração das regras do RNU e substituição do indicador antigo. O SNS alerta que os valores anteriores e posteriores não são diretamente comparáveis. | Terminar a análise desta série em junho de 2026 e manter o novo indicador separado. |

A consulta preliminar da API de **13 de outubro de 2026** encontrou **55 unidades em dezembro de 2023** e **39 em janeiro de 2024**. Essa alteração coincide com a reorganização institucional. A contagem será reproduzida e apresentada no notebook de entendimento dos dados.

O caso da **ULS da Região de Leiria** mostra por que a mudança não se resume a nomes: a ULS integrou o ACES de Pinhal Litoral e centros de saúde que pertenciam aos ACES do Médio Tejo e do Oeste Norte. Uma subida de inscritos na passagem de ACES para ULS poderia resultar da área abrangida, e não de uma evolução no mesmo território.

**Decisão D01:** usar janeiro de 2024 a junho de 2026 na análise principal por unidade e nas previsões históricas. Conservar os dados anteriores como parte da recolha original e como evidência da decisão de recorte. Não reconstruir automaticamente séries antigas das atuais ULS.

Na verificação preliminar, o histórico tinha **6 450 registos** e o recorte **1 170**, em **30 meses**. Os **5 280 registos anteriores a 2024** ficam fora da análise principal por uma decisão de âmbito, não por serem considerados errados. Estes totais serão novamente conferidos na recolha e no tratamento.

O período escolhido também exige verificação de continuidade: manter 39 unidades por mês não prova que todos os nomes e limites territoriais se mantiveram iguais.

### 1.3. Registo da decisão nas fases seguintes

| Fase CRISP-DM | O que fica documentado |
|---|---|
| Entendimento do Negócio | A justificação do período, a decisão D01 e os limites do produto. |
| Entendimento dos Dados | As contagens de unidades por mês, alterações de nomes, informação territorial e fontes institucionais. |
| Preparação dos Dados | O filtro temporal, as quantidades antes e depois, o histórico preservado e as correspondências de nomes verificadas. |

Esta decisão resulta da exploração da fonte e tem de ser rastreável no tratamento. Os dados brutos serão guardados em `data/input`; os dados preparados em `data/output`; as tabelas de verificação em `data/analytics`; as decisões e fontes serão também registadas em `docs`.

**Fontes:** [metadados e alerta do SNS](https://transparencia.sns.gov.pt/api/explore/v2.1/catalog/datasets/utentes-inscritos-em-cuidados-de-saude-primarios), [Alerta de supervisão n.º 1/2024 da ERS](https://www.ers.pt/pt/atividade/supervisao/selecionar/alertas-de-supervisao/2024/alertas/alerta-de-supervis%C3%A3o-n-%C2%BA-1-2024/) e [Relatório e Contas de 2023 da ULS da Região de Leiria](https://www.ulsrl.min-saude.pt/wp-content/uploads/sites/38/2025/06/RC_2023_compressed.pdf).

## 2. Problema de negócio

**A pergunta central é: onde existem maiores falhas de cobertura, quais estão a agravar-se e que informação pode ajudar a acompanhá-las?**

A decisão que se pretende apoiar é a escolha de unidades ou regiões para uma análise mais aprofundada. Para isso, o projeto deve distinguir três situações:

- unidades com mais pessoas sem médico atribuído;
- unidades com maior proporção de inscritos sem médico;
- unidades cuja cobertura está a deteriorar-se ao longo do tempo.

Os resultados serão apresentados em tabelas e gráficos comparáveis, com indicação do período, dos critérios e das limitações. Não serão usados para determinar automaticamente a distribuição de recursos ou o número de médicos a contratar, porque a fonte escolhida não fornece a informação necessária para essas decisões.

Não existe uma entidade cliente ou uma utilização operacional confirmada. O produto inicial é uma análise independente; os destinatários indicados são utilizadores potenciais dos resultados.

## 3. Questões de negócio

| ID | Pergunta | Prioridade | Utilidade |
|---|---|---|---|
| Q1 | Que unidades têm mais inscritos sem médico em junho de 2026, último mês da série? | Alta | Identificar onde se concentra o maior número de pessoas afetadas. |
| Q2 | Que unidades apresentam maior percentagem de inscritos sem médico e como se distribuem por região? | Alta | Comparar cobertura entre unidades de dimensões diferentes. |
| Q3 | Que unidades melhoraram ou pioraram durante o período? | Alta | Identificar tendências e situações que merecem acompanhamento. |
| Q4 | Como evoluiu a percentagem sem médico por região de saúde? | Alta | Apresentar uma visão territorial com percentagens calculadas a partir dos totais. |
| Q5 | Existe associação entre a percentagem sem médico e a utilização de consultas? | Média | Verificar se os indicadores se relacionam, sem atribuir causalidade. |
| Q6 | Na série histórica de janeiro de 2024 a junho de 2026, o histórico permite estimar o valor observado no mês seguinte melhor do que repetir o último valor? | Média | Avaliar previsões históricas dentro da mesma definição do indicador. |

A resposta às primeiras cinco perguntas constitui o produto principal. A previsão depende da qualidade, continuidade e dimensão do histórico disponível.

A comparação geográfica da Q2 será aprofundada pela H4, considerando unidades de dimensão semelhante. A Q5 permanece uma análise secundária e depende da confirmação das definições das taxas de utilização de consultas.

## 4. Hipóteses a verificar

As hipóteses orientam a exploração. Não são conclusões nem explicações já demonstradas.

| ID | Hipótese | Indicadores relacionados | Evidência a procurar |
|---|---|---|---|
| H1 | O ranking por número de inscritos sem médico difere do ranking por percentagem. | Inscritos, número e percentagem sem médico | Comparação dos dois rankings no mesmo mês. |
| H2 | A evolução da cobertura não é uniforme entre unidades e regiões. | Percentagem sem médico e período | Trajetórias e variações entre períodos comparáveis. |
| H3 | Um aumento dos inscritos pode estar associado a um aumento da percentagem sem médico. | Variação dos inscritos e da percentagem sem médico | Relação entre variações, depois de verificar alterações de território e de registo. |
| H4 | Unidades com números semelhantes de inscritos apresentam percentagens sem médico diferentes consoante a região de saúde. | Região de saúde, total de inscritos e percentagem sem médico | Comparar percentagens entre regiões dentro de faixas comuns de dimensão das unidades, no mesmo mês; verificar a consistência dessa diferença ao longo do período. |

**Foco geográfico da H4:** a comparação distingue localização e dimensão. As faixas de inscritos serão comuns a todas as regiões e definidas a partir da dimensão das unidades, antes de observar as diferenças de percentagem. A comparação será feita no mesmo mês, apenas onde existirem unidades comparáveis. Serão apresentados os tamanhos dos grupos e as limitações de grupos pequenos. A hipótese pode não ser sustentada pelos dados; mesmo que existam diferenças, isso não demonstra que a localização seja a causa.

A disponibilidade de médicos, as alterações administrativas e outros fatores poderão influenciar os indicadores. Como nem todos são observados nesta fonte, não será possível atribuir-lhes efeitos com esta análise.

A H2 procura diferenças na evolução temporal; a H4 procura diferenças geográficas entre unidades de dimensão semelhante. Uma hipótese sem suporte, ou sem dados suficientes para uma comparação, será identificada como tal.

## 5. Objetivos analíticos

### Objetivo principal

Construir uma análise da cobertura que permita comparar unidades, acompanhar a evolução e distinguir a dimensão do problema da sua proporção entre os inscritos.

### Componente preditiva

Avaliar historicamente uma previsão numérica da percentagem sem médico para o mês seguinte, por unidade, apenas em pares de meses abrangidos pela série anterior a julho de 2026. A decisão sobre a sua utilidade será tomada depois de comparar o erro com uma referência simples.

| Aspeto | Definição |
|---|---|
| Tipo de análise | Descritiva e temporal, com uma componente de previsão numérica |
| Unidade de análise | Uma unidade de saúde num mês |
| Indicador principal | Percentagem de inscritos sem médico atribuído, mantendo separada a categoria por opção |
| Variável-alvo | Valor desse indicador no mês seguinte, para a mesma unidade |
| Horizonte | Um mês dentro da série histórica de 2024 a junho de 2026; escolha alinhada com a periodicidade mensal |
| Estrutura | Várias observações da mesma unidade, ordenadas no tempo |
| Informação utilizável | Valores do mês de origem e anteriores, conhecidos no momento definido para a previsão |

O número absoluto sem médico será usado para medir a dimensão da situação. A percentagem será usada para comparar a cobertura. A categoria **sem médico por opção** será analisada separadamente, segundo as definições da fonte.

### Momento e alcance da previsão

Uma previsão usa os dados de um mês já divulgado para estimar o indicador da mesma unidade no mês seguinte. Não usa valores do período que procura prever.

A componente preditiva será uma **avaliação histórica de previsões a um mês**, com períodos de teste até junho de 2026. Não serão ligadas previsões de junho a observações de julho de 2026, porque existe uma quebra de definição nessa transição. A análise não constitui uma previsão operacional atual do novo indicador.

Se não houver versões históricas das publicações, o teste será feito sobre a série atualmente disponível. Isso não reproduz exatamente a informação que estava acessível em cada data, sobretudo quando os dados foram revistos. Esta limitação ficará explícita nos resultados.

## 6. Métricas de sucesso

### Acompanhamento da cobertura

- **Número de inscritos sem médico:** dimensão da situação na unidade e no mês.
- **Percentagem sem médico:** proporção da categoria analisada no total de inscritos.
- **Variação em pontos percentuais:** diferença da cobertura entre períodos comparáveis.

As percentagens regionais serão calculadas com a soma das contagens e dos inscritos. Não será usada uma média simples das percentagens das unidades. Os números de diferentes meses não serão somados como se representassem pessoas distintas.

### Qualidade da previsão

A métrica principal será o **erro absoluto médio (MAE), em pontos percentuais**. Permite interpretar diretamente a distância média entre o valor previsto e o observado. Por exemplo, um MAE de 1,2 representa um erro absoluto médio de 1,2 pontos percentuais.

A referência será prever que o valor do mês seguinte é igual ao último valor observado da unidade. A comparação será feita para os mesmos meses e unidades.

Serão também apresentados o erro por unidade e a tendência para sobrestimar ou subestimar o indicador. Subestimar pode ocultar uma deterioração; sobrestimar pode gerar atenção desnecessária. A importância operacional desses erros teria de ser acordada com quem viesse a utilizar os resultados.

## 7. Critérios de sucesso

| Dimensão | Critério verificável |
|---|---|
| Resposta ao problema | Q1 a Q4 respondidas com evidência; Q5 analisada se a definição das taxas o permitir. H1 a H4 avaliadas, incluindo conclusões de ausência de suporte ou de dados suficientes. |
| Qualidade dos dados | Todos os registos usados têm unidade e período válidos; duplicados e valores em falta são identificados e tratados com decisões documentadas. |
| Comparabilidade | Decisão D01 reproduzida com filtro e contagens; mudanças dentro do recorte verificadas. Comparações sem correspondência fiável são excluídas e documentadas. |
| Reprodutibilidade | A análise pode ser executada do início ao fim sobre a recolha guardada, com fonte, filtros e data documentados. |
| Previsão | O erro médio é inferior ao da referência simples num período posterior ao utilizado para desenvolver o modelo. |
| Interpretação | São apresentados o ganho em pontos percentuais, os erros por unidade e os limites da comparação. |

Ter um erro inferior ao da referência é o requisito mínimo para considerar que a previsão acrescentou informação neste teste. Não demonstra, por si só, utilidade operacional: um ganho muito pequeno ou irregular pode não justificar o uso de um modelo mais complexo.

Se a previsão não superar a referência, será mantida a abordagem simples e essa conclusão será apresentada. O valor da análise descritiva será avaliado separadamente do desempenho da previsão.

## 8. Recursos, riscos e restrições

### Recursos e âmbito inicial

- Dados públicos agregados do indicador antigo do Portal da Transparência do SNS. Recolha do histórico de 2016 a junho de 2026 pela API oficial e análise principal de 2024 a junho de 2026.
- Python e notebooks para recolha, preparação, análise e apresentação dos resultados.
- Execução inicial planeada para cerca de 16 a 18 horas, com margem para correções.
- Uma fonte principal. O cruzamento com dados populacionais só será acrescentado se responder a uma pergunta específica e houver correspondência territorial e temporal.
- Scraping reservado para uma extensão de recolha, se for permitido e houver tempo; não condiciona o produto principal.

### Riscos que condicionam a interpretação

| Risco | Resposta prevista |
|---|---|
| Reorganização ACES/ULS em janeiro de 2024 | Usar janeiro de 2024 a junho de 2026 nas comparações por unidade; conservar e documentar o histórico excluído do recorte. |
| Alterações de nomes ou limites dentro do recorte | Verificar correspondências antes de ligar séries; restringir a comparação quando necessário. |
| Revisões dos dados e atraso de publicação | Guardar a recolha original e identificar o período analisado e a data de recolha. |
| Descontinuação e quebra de série em julho de 2026 | Manter a análise histórica até junho de 2026; não juntar os indicadores antigos e novos como uma única série. |
| Poucos meses de histórico | Limitar a complexidade da previsão e reconhecer a incerteza dos resultados. |
| Informação do futuro usada no modelo | Definir o momento da previsão e restringir os dados ao que já é conhecido nesse momento. |
| Inscritos confundidos com população residente | Usar os inscritos como universo do indicador e não extrapolar para todos os habitantes. |
| Associação confundida com causa | Apresentar relações observadas e fatores não medidos; evitar explicações causais. |
| Taxas de consultas com janelas de um ano sobrepostas | Considerar que meses consecutivos partilham parte da informação e não são observações independentes. |

A recolha ficará limitada aos dados públicos necessários à análise. As condições de utilização da fonte serão respeitadas, os conteúdos brutos serão preservados e as transformações documentadas.

### Organização dos ficheiros

| Local | Conteúdo |
|---|---|
| `notebooks/` | Um notebook por fase CRISP-DM, com decisões, execução e interpretação dos resultados. |
| `data/input/` | Recolhas brutas e respetivos metadados, preservados sem alterações. |
| `data/output/` | Dataset preparado e previsões exportadas. |
| `data/analytics/` | Gráficos, tabelas de análise, verificações da qualidade e métricas. |
| `data/models/` | Modelos treinados e elementos necessários para reutilizar a previsão. |
| `src/` | Funções reutilizáveis de recolha e processamento. |
| `docs/` | Documentação, fontes e registo das decisões do projeto. |
| `media/` | Imagens de apoio à apresentação. |
| `README.md` | Objetivo, organização e instruções para executar o projeto. |
| `.gitignore` | Regras para excluir ficheiros temporários e outros elementos sem utilidade no repositório. |

Os notebooks são `01_business_understanding.ipynb`, `02_data_understanding.ipynb`, `03_data_preparation.ipynb`, `04_modeling.ipynb`, `05_evaluation.ipynb` e `06_deployment.ipynb`, todos dentro de `notebooks/`.

Os caminhos usados no código serão definidos em relação à raiz do projeto, para funcionarem com esta estrutura e sem dependerem de caminhos pessoais do computador.

## 9. Impacto esperado

O resultado esperado é uma visão comparável da cobertura de médico de família, que facilite localizar situações com maior número de inscritos afetados, menor cobertura proporcional ou tendência de deterioração.

A análise pode ajudar a orientar uma investigação posterior das unidades identificadas e a acompanhar os indicadores ao longo do tempo. A previsão, se acrescentar informação, poderá complementar esse acompanhamento.

Estes são benefícios esperados, não resultados já demonstrados. Qualquer utilização para planeamento de recursos exigiria informação adicional, incluindo capacidade assistencial, disponibilidade de profissionais e contexto local.

O produto será disponibilizado em notebooks, tabelas e gráficos exportados, acompanhados dos dados de origem, dos dados preparados e das decisões de tratamento. Os resultados deverão ser interpretáveis sem depender da leitura de todo o código.

## 10. Síntese do enquadramento

O projeto analisa historicamente a cobertura de médico de família entre os inscritos nos cuidados de saúde primários de Portugal Continental, de janeiro de 2024 a junho de 2026.
A decisão apoiada é a identificação de unidades e regiões para acompanhamento e investigação mais aprofundados.
A análise combina o número de inscritos sem médico, a sua percentagem e a evolução em períodos comparáveis.
A categoria sem médico por opção será mantida separada.
A componente preditiva avalia estimativas do mês seguinte por unidade, usando apenas meses anteriores e mantendo o teste dentro da série até junho de 2026.
O erro será medido em pontos percentuais e comparado com a referência de repetir o último valor.
O resultado será uma análise reproduzível, com evidência, decisões documentadas e limitações explícitas.

### Passagem para o entendimento dos dados

No notebook `02_data_understanding.ipynb` serão recolhidos e preservados os dados brutos em `data/input`. Serão confirmados o esquema, as definições dos indicadores, a cobertura temporal, a qualidade dos valores e a continuidade das unidades. A mudança de 55 para 39 unidades será demonstrada com dados e ligada às fontes institucionais. O recorte e as correspondências verificadas serão implementados em `03_data_preparation.ipynb`. O âmbito ou as hipóteses poderão ser ajustados se essa verificação revelar limitações.

[Continuar para 02_data_understanding.ipynb](02_data_understanding.ipynb)